# Plotting the results of a simulation for SKA mid and low in the AA4 configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
colours = ["#FF5F15", "#0047AB", "black"]

## Loading the observed pulsar population

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]["(s)"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf[("ASSOC", "Unnamed: 55_level_1")].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
df_atnf.head()

In [ ]:
df_atnf.columns = df_atnf.columns.droplevel(1)
df_atnf.head()

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PMRA",
        "PMDEC",
        "PX",
        "POSEPOCH",
        "RAJD",
        "DECJD",
        "DM",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "XX",
        "YY",
    ],
)

len(df_atnf)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

Note: For the purpose of modelling the observed isolated population of radio pulsars, in the following, we count those objects with period derivatives larger than $\dot{P} > 10^{-19} s/s$ or those with no measured period derivatives. The latter are likely isolated in nature due to the fact that only a small fraction of ATNF pulsars with known $\dot{P}$ actually have been recycled and attain $\dot{P} < 10^{-19} s/s$. As a result, the following number counts are slighlty larger than the populations used to produce our period period-derivative maps for the simulation-based inference approach.

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Extract periods, period derivatives, longitude and latitude for all remaining pulsars.
P_obs = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_obs = df_atnf["P1"].to_numpy().astype(np.float64)
l_obs = df_atnf["Gl"].to_numpy().astype(np.float64)
b_obs = df_atnf["Gb"].to_numpy().astype(np.float64)

In [ ]:
l_obs[(l_obs > 180.0) & (l_obs < 360.0)] = (
    l_obs[(l_obs > 180.0) & (l_obs < 360.0)] - 360.0
)

In [ ]:
print(len(P_obs))

## Loading the simulated pulsar populations

In [ ]:
dir = "SKA_AA4_br2_2e9yrs"

data_full = pd.read_pickle(
    f"../../test/{dir}/final_population.pkl.gz",
    compression="gzip",
)

data_PMPS = pd.read_pickle(
    f"../../test/{dir}/survey_PMPS_results.pkl.gz",
    compression="gzip",
)


data_SMPS = pd.read_pickle(
    f"../../test/{dir}/survey_SMPS_results.pkl.gz",
    compression="gzip",
)

data_HTRU_low_mid = pd.read_pickle(
    f"../../test/{dir}/survey_HTRU_low_mid_results.pkl.gz",
    compression="gzip",
)

data_HTRU_high = pd.read_pickle(
    f"../../test/{dir}/survey_HTRU_high_results.pkl.gz",
    compression="gzip",
)

data_SKA_low = pd.read_pickle(
    f"../../test/{dir}/survey_SKA_low_results.pkl.gz",
    compression="gzip",
)

data_SKA_mid = pd.read_pickle(
    f"../../test/{dir}/survey_SKA_mid_results.pkl.gz",
    compression="gzip",
)

data_SKA_low.head()

In [ ]:
x = data_full["x"]["[kpc]"].to_numpy()
y = data_full["y"]["[kpc]"].to_numpy()
z = data_full["z"]["[kpc]"].to_numpy()
RA = data_full["RA"]["[deg]"].to_numpy()
DEC = data_full["DEC"]["[deg]"].to_numpy()
pm_RA = data_full["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = data_full["pm_DEC"]["[mas yr^-1]"].to_numpy()
l = data_full["l"]["[deg]"].to_numpy()
b = data_full["b"]["[deg]"].to_numpy()
v_r = data_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = data_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = data_full["v_z"]["[km s^-1]"].to_numpy()
dist = data_full["d"]["[kpc]"].to_numpy()
B = data_full["B"]["[G]"].to_numpy()
chi = data_full["chi"]["[rad]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_full["w_int"]["[s]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)
age = data_full["age"]["[yr]"].to_numpy()

In [ ]:
idx_PMPS = data_PMPS["NS_idx"][" "].to_numpy(dtype=int)
idx_SMPS = data_SMPS["NS_idx"][" "].to_numpy(dtype=int)
idx_HTRU_low_mid = data_HTRU_low_mid["NS_idx"][" "].to_numpy(dtype=int)
idx_HTRU_high = data_HTRU_high["NS_idx"][" "].to_numpy(dtype=int)

idx_SKA_low = data_SKA_low["NS_idx"][" "].to_numpy(dtype=int)
idx_SKA_mid = data_SKA_mid["NS_idx"][" "].to_numpy(dtype=int)

set_PMPS = set(idx_PMPS.tolist())
set_SMPS = set(idx_SMPS.tolist())
set_HTRU_low_mid = idx_HTRU_low_mid.tolist()
set_HTRU_high = idx_HTRU_high.tolist()
set_SKA_low = set(idx_SKA_low.tolist())
set_SKA_mid = set(idx_SKA_mid.tolist())

idx_SMPS_noduplicates = list(set_SMPS - set_PMPS)

idx_radio_det = idx_PMPS.tolist() + idx_SMPS_noduplicates

In [ ]:
P_HTRU_high = data_HTRU_high["P"]["[s]"].to_numpy()
print(P_HTRU_high[:10])
print(P[idx_HTRU_high][:10])

In [ ]:
P_SKA_low = data_SKA_low["P"]["[s]"].to_numpy()
print(P_SKA_low[:10])
print(P[idx_SKA_low][:10])

In [ ]:
P_SKA_mid = data_SKA_mid["P"]["[s]"].to_numpy()
print(P_SKA_mid[:10])
print(P[idx_SKA_mid][:10])

A few statistics on the detected number of stars.

In [ ]:
number_intercepted = len(intercepted_radio[intercepted_radio == True])
number_detected_PMPS = len(idx_PMPS)
number_detected_SMPS = len(idx_SMPS)
number_detected_HTRU_low_mid = len(idx_HTRU_low_mid)
number_detected_HTRU_high = len(idx_HTRU_high)
number_detected_SKA_low = len(idx_SKA_low)
number_detected_SKA_mid = len(idx_SKA_mid)

In [ ]:
print(
    f"Total number of detected pulsars without SKA: {number_detected_PMPS + number_detected_SMPS + number_detected_HTRU_low_mid + number_detected_HTRU_high}"
)
print(f"Detected pulsars with PMPS: {number_detected_PMPS}")
print(f"Detected pulsars with SMPS: {number_detected_SMPS}")
print(
    f"Detected pulsars with HTRU mid and low: {number_detected_HTRU_low_mid}"
)
print(
    f"Total number of detected pulsars with SKA: {number_detected_SKA_low + number_detected_SKA_mid}"
)
print(f"Detected pulsars with SKA low: {number_detected_SKA_low}")
print(f"Detected pulsars with SKA mid: {number_detected_SKA_mid}")

## Plotting the positional distributions

Top view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)

ax.plot(
    x[intercepted_radio],
    y[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_PMPS],
    y[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    x[idx_SMPS],
    y[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    x[idx_HTRU_low_mid],
    y[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:purple",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    x[idx_HTRU_high],
    y[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)
ax.plot(
    x[idx_SKA_low],
    y[idx_SKA_low],
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA low",
)
ax.plot(
    x[idx_SKA_mid],
    y[idx_SKA_mid],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA mid",
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=20)

plt.show()

Side view of the Glactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)

ax.plot(
    x[intercepted_radio],
    z[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_PMPS],
    z[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    x[idx_SKA_low],
    z[idx_SKA_low],
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA low",
)
ax.plot(
    x[idx_SKA_mid],
    z[idx_SKA_mid],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA mid",
)
ax.plot(
    x[idx_SMPS],
    z[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    x[idx_HTRU_low_mid],
    z[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:purple",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    x[idx_HTRU_high],
    z[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=20)

plt.show()

## Plotting the distribution along z

In [ ]:
bins_z_distribution = np.linspace(0, 20, 101)

z_values = np.linspace(0, 20, 501)
scale_height_PSRPopPY = 0.33
pdf_z_PSRPopPY = (
    1.0 / scale_height_PSRPopPY * np.exp(-z_values / scale_height_PSRPopPY)
)

In [ ]:
z_observed = df_atnf["ZZ"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    z[idx_HTRU_low_mid],
    bins=bins_z_distribution,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Simulated HTRU low and mid",
    density=True,
)

ax.hist(
    z[idx_SMPS],
    bins=bins_z_distribution,
    histtype="step",
    color="grey",
    lw=4,
    alpha=0.8,
    label=r"Simulated SMPS",
    density=True,
)

ax.hist(
    z_observed,
    bins=bins_z_distribution,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"Observed full",
    density=True,
)

ax.plot(
    z_values,
    pdf_z_PSRPopPY,
    color="orange",
    lw=4,
    alpha=0.8,
    label=r"Exp. PDF PSRPopPy $h_c = 0.33\,$kpc",
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"Number of stars detected")
ax.set_xlim(0.0, 5.0)
ax.set_ylim(0.0, 3.0)
plt.legend(frameon=True, loc=1)

plt.show()

Make histogram for all the SKA detections and fit an exponential PDF.

In [ ]:
z_obs_combined = np.append(z[idx_SKA_low], z[idx_SKA_mid])
print(len(z_obs_combined))

In [ ]:
from scipy.optimize import curve_fit

# Create histogrammed data points as for the plot with density equals True.
hist_values, _ = np.histogram(
    z_obs_combined, bins=bins_z_distribution, density=True
)

# Compute bin centers for the fit.
bin_centers = (bins_z_distribution[:-1] + bins_z_distribution[1:]) / 2

print(hist_values)
print(bin_centers)

In [ ]:
# Define the exponential function to be fitted.
def exp_pdf(x, lamb):
    return (1 / lamb) * np.exp(-x / lamb)


# Fit function to the histogrammed data.
fit, _ = curve_fit(exp_pdf, bin_centers, hist_values)

lambda_fit = fit[0]
print(lambda_fit)

In [ ]:
scale_height_low = np.round(lambda_fit, 2)
pdf_z_low = 1.0 / scale_height_low * np.exp(-z_values / scale_height_low)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    z[idx_SKA_low],
    bins=bins_z_distribution,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA low",
    density=True,
)

ax.hist(
    z[idx_SKA_mid],
    bins=bins_z_distribution,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA mid",
    density=True,
)

ax.hist(
    z_obs_combined,
    bins=bins_z_distribution,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA combined",
    density=True,
)

ax.plot(
    z_values,
    pdf_z_PSRPopPY,
    color="grey",
    lw=4,
    alpha=0.8,
    label=r"Exp. PDF PSRPopPy $h_c = 0.33\,$kpc",
)

ax.plot(
    z_values,
    pdf_z_low,
    color=colours[2],
    lw=4,
    alpha=0.8,
    label=f"Exp. PDF $h_c = {scale_height_low}\,$kpc",
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"Stars detected")
ax.set_xlim(0.0, 20.0)
ax.set_ylim(0.0, 0.6)
plt.legend(frameon=True, loc=1)

plt.show()

## Plotting the positional distribution

In [ ]:
l_SKA_low = data_SKA_low["l"]["[deg]"].to_numpy()
b_SKA_low = data_SKA_low["b"]["[deg]"].to_numpy()
l_SKA_mid = data_SKA_mid["l"]["[deg]"].to_numpy()
b_SKA_mid = data_SKA_mid["b"]["[deg]"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

plt.plot(
    l_SKA_low,
    b_SKA_low,
    linestyle="None",
    marker="o",
    color="orange",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Simulated SKA low",
)

plt.plot(
    l_SKA_mid,
    b_SKA_mid,
    linestyle="None",
    marker="o",
    color=colours[0],
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Simulated SKA mid",
)

plt.plot(
    l_obs,
    b_obs,
    linestyle="None",
    marker="o",
    color=colours[1],
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Observed population",
)

ax.set_xlim(-190.0, 190.0)
ax.set_ylim(-95.0, 95.0)
ax.set_xlabel(r"Galactic longitude $l$ [deg]")
ax.set_ylabel(r"Galactic latitude $b$ [deg]")
ax.legend(frameon=True, loc="best")
ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
# Convert Galactic longitude to radians for the Aitoff projection.
l_obs_rad = np.radians(l_obs)
l_SKA_low_rad = np.radians(l_SKA_low)
l_SKA_mid_rad = np.radians(l_SKA_mid)

# Convert Galactic latitude to radians.
b_obs_rad = np.radians(b_obs)
b_SKA_low_rad = np.radians(b_SKA_low)
b_SKA_mid_rad = np.radians(b_SKA_mid)

fig, ax = plt.subplots(figsize=(15, 10), subplot_kw={"projection": "aitoff"})

ax.scatter(
    l_SKA_low_rad,
    b_SKA_low_rad,
    marker="o",
    color="orange",
    s=40,
    alpha=1,
    label=r"Simulated SKA low",
)

ax.scatter(
    l_SKA_mid_rad,
    b_SKA_mid_rad,
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"Simulated SKA mid",
)

ax.scatter(
    l_obs_rad,
    b_obs_rad,
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"Observed population",
)

ax.set_xticks(np.radians(np.linspace(-180, 180, 13)))
ax.set_xticklabels(
    [
        "",
        "",
        r"$-120^\circ$",
        "",
        r"$-60^\circ$",
        "",
        r"$0^\circ$",
        "",
        r"$60^\circ$",
        "",
        r"$120^\circ$",
        "",
        "",
    ]
)
ax.set_yticks(np.radians(np.linspace(-90, 90, 7)))
ax.set_yticklabels(
    [
        r"$-90^\circ$",
        r"$-60^\circ$",
        r"$-30^\circ$",
        r"$0^\circ$",
        r"$30^\circ$",
        r"$60^\circ$",
        r"$90^\circ$",
    ]
)

ax.grid(True)  # Add grid lines
ax.legend(frameon=True, bbox_to_anchor=(0.7, 0.95))

plt.tight_layout()
plt.show()

## Plotting the DM distribution

In [ ]:
DM_SKA_mid = data_SKA_mid["DM"]["[pc cm^-3]"].to_numpy()
DM_SKA_low = data_SKA_low["DM"]["[pc cm^-3]"].to_numpy()
P_SKA_low = data_SKA_low["P"]["[s]"].to_numpy()
P_SKA_mid = data_SKA_mid["P"]["[s]"].to_numpy()
P_dot_SKA_low = data_SKA_low["P_dot"]["[s s^-1]"].to_numpy()
P_dot_SKA_mid = data_SKA_mid["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

ax.scatter(
    P_SKA_mid,
    DM_SKA_mid,
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"Simulated SKA mid",
)

ax.scatter(
    P_SKA_low,
    DM_SKA_low,
    marker="o",
    color="orange",
    s=40,
    alpha=1,
    label=r"Simulated SKA low",
)

ax.set_xscale("log")

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"DM [pc/cm$^3$]")
plt.legend(frameon=True, loc=1)

plt.show()

## Plotting a PPdot diagram and corresponding histograms

Edot lines.

In [ ]:
I_NS = 1.36e45
R_NS = 1.1e6  # in cm.
M_NS = 1.4  # in solar masses.
c = 2.998e10

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * c**3)
    )

Deathlines.

In [ ]:
P_dot_DL_1 = (
    1.2e-15
    * (1.4 / 1) ** (-1)
    * (R_NS / 1e6) ** (-3 / 4)
    * (10**P_log) ** (11 / 4)
)
b = 10
P_dot_DL_2 = 2.1e-18 * (1.4 / 1) ** (-1) * b**0.5 * (10**P_log) ** 2

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))
ax.set_title(f"SKA AA4 baseline", fontsize=MEDIUM_SIZE)

# ax.scatter(
#     P,
#     P_dot,
#     marker=".",
#     color="gray",
#     s=40,
#     alpha=1,
#     label=r"Simulated population",
# )
for i in range(len(Edot_log)):
    ax.plot(
        10**P_log,
        Pdot_Edot_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
for i in range(len(B_log)):
    ax.plot(
        10**P_log,
        Pdot_B_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )

ax.plot(
    10**P_log,
    P_dot_DL_1,
    linestyle="-",
    linewidth=3.0,
    color="black",
    alpha=0.7,
    rasterized=True,
)
ax.plot(
    10**P_log,
    P_dot_DL_2,
    linestyle="-",
    linewidth=3.0,
    color="black",
    alpha=0.7,
    rasterized=True,
)
ax.fill_between(10**P_log, P_dot_DL_1, P_dot_DL_2, color="grey", alpha=0.3)

ax.scatter(
    P[idx_SKA_low],
    P_dot[idx_SKA_low],
    marker="o",
    color="orange",
    s=40,
    alpha=1,
    label=r"SKA low simulated",
)

ax.scatter(
    P[idx_SKA_mid],
    P_dot[idx_SKA_mid],
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"SKA mid simulated",
)

ax.scatter(
    P_obs,
    Pdot_obs,
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"Observed",
)

ax.text(
    0.412,
    0.015,
    r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.275,
    0.015,
    r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.143,
    0.015,
    r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.015,
    r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.17,
    r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.325,
    r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.text(
    0.9,
    0.027,
    r"$10^{10} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.182,
    r"$10^{11} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.335,
    r"$10^{12} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.49,
    r"$10^{13} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.645,
    r"$10^{14} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1.2e-3, 100.0)
ax.set_ylim(1.0e-22, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(frameon=True, loc=2)

plt.tight_layout()
plt.savefig("SKA_ppdot_AA4.pdf", dpi=300, bbox_inches="tight")
plt.show()

Observed and SKA numbers above lower deathline.

In [ ]:
P_dot_DL_obs_cutoff = 2.1e-18 * (1.4 / 1) ** (-1) * b**0.5 * (P_obs) ** 2
print(sum(Pdot_obs < P_dot_DL_obs_cutoff))  # Below as a check

P_dot_DL_SKA_low_cutoff = (
    2.1e-18 * (1.4 / 1) ** (-1) * b**0.5 * (P_SKA_low) ** 2
)
print(sum(P_dot_SKA_low > P_dot_DL_SKA_low_cutoff))

P_dot_DL_SKA_mid_cutoff = (
    2.1e-18 * (1.4 / 1) ** (-1) * b**0.5 * (P_SKA_mid) ** 2
)
print(sum(P_dot_SKA_mid > P_dot_DL_SKA_mid_cutoff))

In [ ]:
# Define bins for the histogram x-axes.
bins_period = np.linspace(-2.4, 2.4, 25)
print(bins_period)

bins_period_deriv = np.linspace(-22, -10, 21)
print(bins_period_deriv)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(P_obs),
    bins=bins_period,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"Observed population",
)

ax.hist(
    np.log10(P[idx_SKA_mid]),
    bins=bins_period,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA mid",
)

ax.hist(
    np.log10(P[idx_SKA_low]),
    bins=bins_period,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA low",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10} P$ [s]")
plt.ylabel(r"Number of stars detected")
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
bins_period_B = np.linspace(1e-1, 10, 55)
P_values = np.linspace(1e-1, 10, 155)
mean_PDF = 0.3
std_PDF = 0.15
PDF_P_PSRPopPy = (
    1
    / (std_PDF * np.sqrt(2 * np.pi**2))
    * np.exp(-((P_values - mean_PDF) ** 2) / ((2 * std_PDF) ** 2))
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    P,
    bins=bins_period_B,
    histtype="step",
    color="gray",
    lw=4,
    alpha=0.8,
    density=True,
    label=r"Full simulated population",
)

ax.hist(
    P_obs,
    bins=bins_period_B,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    density=True,
    label=r"Observed population",
)

ax.hist(
    P[idx_SKA_mid],
    bins=bins_period_B,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    density=True,
    label=r"Simulated SKA mid",
)

ax.hist(
    P[idx_SKA_low],
    bins=bins_period_B,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.8,
    density=True,
    label=r"Simulated SKA low",
)

ax.plot(
    P_values,
    PDF_P_PSRPopPy,
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Normal PDF PSRPopPy",
)

ax.set_xscale("log")

plt.xlabel(r"log$_{10} P$ [s]")
plt.ylabel(r"Number of stars detected")
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(Pdot_obs),
    bins=bins_period_deriv,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"Observed population",
)

ax.hist(
    np.log10(P_dot[idx_SKA_mid]),
    bins=bins_period_deriv,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA mid",
)

ax.hist(
    np.log10(P_dot[idx_SKA_low]),
    bins=bins_period_deriv,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.8,
    label=r"Simulated SKA low",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10} \dot{P}$ [s/s]")
plt.ylabel(r"Number of stars detected")
plt.legend(frameon=True, loc=1)

plt.show()